# GitHub API

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `github_api.ipynb`, `other_sources.ipynb`)._

In [ ]:
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # picks up GITHUB_TOKEN from .env if you have one (optional, see below)

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "decay-before-archival",  # required by the GitHub API
})

token = os.environ.get("GITHUB_TOKEN")  # optional: raises rate limits a lot (5000 req/h vs 60)
if token:
    session.headers["Authorization"] = f"Bearer {token}"

print("authenticated" if token else "unauthenticated — search is limited to 10 requests/min, core API to 60/hour")

## Explore: GitHub Search API

The GitHub REST API fills gaps the BigQuery sources don't cover directly — live repo state (stars, forks, open issues/PRs, archived flag, last push) for repos that may not be in deps.dev/Scorecard yet.

**Planned analysis:** build the candidate pool of **open-source repos with 1000+ stars created between 2024 and 2025** (`stars:>=1000 created:2024-01-01..2025-12-31`). GitHub's search API has no license qualifier, so the open-source filter is applied after fetching — keep only repos with an identifiable SPDX license (`license_spdx`); `NOASSERTION` (custom license) and unlicensed repos are dropped by default. These are young-but-popular projects — the cohort we'll later watch for decay signals (stalled releases, contributor turnover, issue backlog growth) before archival.

**Rate limits:** unauthenticated search allows 10 requests/min and the core API 60/hour; a `GITHUB_TOKEN` in `.env` (free account) raises that to 30 searches/min and 5,000 core req/h. The `gh_get` helper below backs off automatically when it hits a limit.

In [ ]:
def gh_get(url, params=None):
    """GET with automatic backoff on rate-limit (403/429). Returns parsed JSON."""
    if "session" not in globals():
        raise RuntimeError(
            "GitHub session is not set up — run the first code cell "
            "(requests.Session + GITHUB_TOKEN) before running this notebook's queries."
        )
    while True:
        resp = session.get(url, params=params, timeout=30)
        if resp.status_code in (403, 429):
            reset = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(reset - time.time(), 1) + 1
            print(f"rate-limited — sleeping {wait:.0f}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()


In [ ]:
# Candidate pool: repos created 2024-2025 with at least 1000 stars, most-starred first
query = "stars:>=1000 created:2024-01-01..2025-12-31"
per_page = 100
max_pages = 3  # cap while exploring; raise once we know how many we actually want

items, page = [], 1
while page <= max_pages:
    data = gh_get("https://api.github.com/search/repositories", params={
        "q": query, "sort": "stars", "order": "desc",
        "per_page": per_page, "page": page,
    })
    print(f"total matching repos: {data['total_count']:,} — fetched page {page}")
    items.extend(data["items"])
    if data.get("incomplete_results"):
        print("warning: GitHub flagged these results as incomplete (query too broad) — narrow the query")
    if len(data["items"]) < per_page:
        break
    page += 1

repos = pd.DataFrame(items)
repos["license_spdx"] = repos["license"].apply(lambda L: L["spdx_id"] if L else None)
repos = repos[[
    "full_name", "stargazers_count", "forks_count", "open_issues_count",
    "language", "created_at", "pushed_at", "archived", "license_spdx", "description",
]].rename(columns={"full_name": "repo_full_name"})
repos["repo_owner"] = repos["repo_full_name"].str.split("/").str[0]
repos["repo_name"] = repos["repo_full_name"].str.split("/").str[1]
repos["created_at"] = pd.to_datetime(repos["created_at"]).dt.date
repos["pushed_at"] = pd.to_datetime(repos["pushed_at"]).dt.date

# GitHub's search API has no license qualifier, so "open source" is filtered here:
#   license_spdx == None        -> no license file at all (all rights reserved — not open source)
#   license_spdx == "NOASSERTION" -> custom/unrecognized license (may still be OSS — check manually)
OPEN_SOURCE_ONLY = True
if OPEN_SOURCE_ONLY:
    n_before = len(repos)
    repos = repos[repos["license_spdx"].notna()]
    print(f"open-source filter: {n_before} -> {len(repos)} rows "
          f"(dropped {n_before - len(repos)} without an identifiable license)")

repos.head(20)

**Notes:**

_(add observations here)_

In [ ]:
# Quick look at the pool: size, year split, languages, and how many are already archived
repos["created_year"] = pd.to_datetime(repos["created_at"]).dt.year
print(f"rows fetched: {len(repos):,}")
print("\nby creation year:")
print(repos["created_year"].value_counts().sort_index())
print("\ntop languages:")
print(repos["language"].value_counts(dropna=False).head(8))
print(f"\nalready archived: {int(repos['archived'].sum())}")

**Notes:**

_(add observations here)_

In [ ]:
# Per-repo detail for one sample repo: contributors + open issues (bus factor, issue backlog)
sample_repo = repos.iloc[0]["repo_full_name"]  # most-starred in the pool; swap in any "owner/name"
print(f"exploring {sample_repo}")

# Contributors (top page of 100 by commit count)
contributors = gh_get(f"https://api.github.com/repos/{sample_repo}/contributors", params={"per_page": 100})
contrib_df = pd.DataFrame(contributors)[["login", "contributions"]]

# Open issues (includes PRs — GitHub counts them together in open_issues_count)
issues = gh_get(f"https://api.github.com/repos/{sample_repo}/issues", params={"per_page": 100, "state": "open"})
issue_df = pd.DataFrame(issues)[["number", "title", "created_at", "pull_request"]]
issue_df["is_pr"] = issue_df["pull_request"].notna()

total_commits = contrib_df["contributions"].sum()
top_share = contrib_df.iloc[0]["contributions"] / total_commits if total_commits else float("nan")
print(f"contributors (top {len(contrib_df)}): {len(contrib_df)}, top contributor share: {top_share:.1%}")
print(f"open issues/PRs (first 100): {len(issue_df)}, of which PRs: {int(issue_df['is_pr'].sum())}")
contrib_df.head(10)

**Notes:**

_(add observations here)_

## Join key for integration

`repo_owner` + `repo_name` (and the full `full_name`/`html_url`) — same owner/name pair used by GH Archive (`payload.repo.owner.name` / `name`) and OpenSSF Scorecard, so this source joins directly on those. Package-based sources (deps.dev, PyPI) join via the repo's package manifest or a name match in the integration step.